# GraphML - Assignment 1 | María Sánchez Domínguez

In [1]:
#I have an initial set of codes that ensure the topologicpy installation and version, as well as setting up the libraries. 
#It also shows the initial OBJ file (simple geometry, no apertures yet) to ensure everything is working.
#That is all part 1., part 2. is the assignment graph generation.

## 1.0. - Installation

In [2]:
pip install topologicpy==0.9.19

Note: you may need to restart the kernel to use updated packages.


## 1.1 Import the needed libraries

In [3]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

e:\Documents\GitHub\graphML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2. Check the TopologicPy Version

In [4]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.19) is OLDER than the latest version (0.9.20) from PyPI. Please consider upgrading to the latest version.


## 1.3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [5]:
renderer = "vscode"

## 1.4. Import the OBJ file

In [ ]:
objects = Topology.ByOBJPath(r"assignment1\obj\Assignment1.obj", selfMerge=True)
cells = Topology.Cells(objects[0])
cc = CellComplex.ByCells(cells)
print("Objects is a list")
print(cc)

Topology.ByOBJPath - Error: the input OBJ path does not exist. Returning None.


TypeError: 'NoneType' object is not subscriptable

## 1.5. Show the geometry

In [ ]:
Topology.Show(cc,
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 2.0. ASSIGNMENT: Relational + Aperture Graph

In [13]:
import os
from pathlib import Path

# Get the notebook's directory
notebook_dir = Path(os.getcwd())
print(f"Working directory: {notebook_dir}")

# Construct relative path from working directory to obj files
obj_dir = notebook_dir / "assignments" / "assignment1" / "obj"
obj_file = obj_dir / "Assignment1.obj"

print(f"Looking for file at: {obj_file}")
print(f"File exists: {obj_file.exists()}")

Working directory: e:\Documents\GitHub\graphML\assignments
Looking for file at: e:\Documents\GitHub\graphML\assignments\assignments\assignment1\obj\Assignment1.obj
File exists: False


## 2.1. Import geometry OBJ as Cell Complex

In [14]:
objects = Topology.ByOBJPath(r"assignment1\obj\Assignment1.obj", selfMerge=True)
cells = Topology.Cells(objects[0])
building = CellComplex.ByCells(cells)
print("Objects is a list")
print(building)

Objects is a list


## 2.1. Import apertures (doors, windows) as OBJ (polylines) to generate faces

In [17]:
# Making sure the polylines from the OBJ file are being loaded correctly and as wires
# I found it harder to use the faces directly so I went with an edge extraction->edge to wire->wire to face method

windowFile = Topology.ByOBJPath(r"assignment1\obj\Assignment1_Windows.obj", selfMerge=True)

# check the file
edges = Topology.Edges(windowFile[0])
print(f"Edges result: {edges}, Type: {type(edges)}")

wires = Topology.Wires(windowFile[0])
print(f"Wires result: {wires}, Type: {type(wires)}")

vertices = Topology.Vertices(windowFile[0])
print(f"Vertices result: {vertices}, Type: {type(vertices)}")

Edges result: [<topologic_core.Edge object at 0x00000170C94BFEF0>, <topologic_core.Edge object at 0x00000170C9459570>, <topologic_core.Edge object at 0x00000170C75AAFB0>, <topologic_core.Edge object at 0x00000170C9558130>, <topologic_core.Edge object at 0x00000170C955BCF0>, <topologic_core.Edge object at 0x00000170C955BCB0>, <topologic_core.Edge object at 0x00000170C955BD30>, <topologic_core.Edge object at 0x00000170C95586F0>, <topologic_core.Edge object at 0x00000170C9558870>, <topologic_core.Edge object at 0x00000170C9559130>, <topologic_core.Edge object at 0x00000170C95589B0>, <topologic_core.Edge object at 0x00000170C9558C30>, <topologic_core.Edge object at 0x00000170C955AD70>, <topologic_core.Edge object at 0x00000170C955A1F0>, <topologic_core.Edge object at 0x00000170C9558AF0>, <topologic_core.Edge object at 0x00000170C9559C30>, <topologic_core.Edge object at 0x00000170C955A530>, <topologic_core.Edge object at 0x00000170C955BE70>, <topologic_core.Edge object at 0x00000170C95581F0

In [18]:
windowFile = Topology.ByOBJPath(r"assignment1\obj\Assignment1_Windows.obj", selfMerge=True)

# Extract wires (the closed polylines)
wires = Topology.Wires(windowFile[0])

# Convert each wire to a face
windows = [Face.ByWire(w) for w in wires]

In [19]:
doorFile = Topology.ByOBJPath(r"assignment1\obj\Assignment1_Doors.obj", selfMerge=True)

# Extract wires (the closed polylines)
wires = Topology.Wires(doorFile[0])

# Convert each wire to a face
doors = [Face.ByWire(w) for w in wires]

## 2.2. Add door/window faces as Apertures to the Cell Complex

In [20]:
#Adding apertures to the cell complex

building = Topology.AddApertures(building, [doors, windows], exclusive=True, subTopologyType="face")

## 2.3. Generate graphs

In [21]:
# I want to display two graphs: one showing cell-to-cell touches and another showing cell-to-aperture touches. 
# I will style them differently to distinguish them visually.

# Graph 1: Cell-to-Cell Touches
g_cells = Graph.BySpatialRelationships(cells, include=["touches"], useInternalVertex=True)

# Graph 2: Cell-to-Aperture Touches
apertures = doors + windows
g_apertures = Graph.BySpatialRelationships(cells + apertures, include=["touches"], useInternalVertex=True)

## 2.4. Graph styling and plotting

In [22]:
# Style cells graph
vertices_cells = Graph.Vertices(g_cells)
for v in vertices_cells:
    d = Dictionary.ByKeysValues(["size", "color"], [20, "red"])
    v = Topology.SetDictionary(v, d)

edges_cells = Graph.Edges(g_cells)
for e in edges_cells:
    d = Dictionary.ByKeysValues(["width", "color"], [8, "black"])
    e = Topology.SetDictionary(e, d)

# Style apertures graph
vertices_apertures = Graph.Vertices(g_apertures)
for v in vertices_apertures:
    d = Dictionary.ByKeysValues(["size", "color"], [10, "cyan"])
    v = Topology.SetDictionary(v, d)

edges_apertures = Graph.Edges(g_apertures)
for e in edges_apertures:
    d = Dictionary.ByKeysValues(["width", "color"], [2, "yellow"])
    e = Topology.SetDictionary(e, d)

# Visualize both graphs together
Topology.Show(building, g_apertures, g_cells,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.2,
              width=600,
              height=600,
              backgroundColor="white",
              renderer=renderer)